# OceanSODA (NOAA) Dataset processing. 
The SODA data is used for the DIC and Alk restoring forces. This PR reduces the resolution, time, and number of variables in order to reduce size of the test data file. 

In [1]:
import xarray as xr
import fsspec
import numpy as np

url = "https://www.ncei.noaa.gov/data/oceans/archive/arc0160/0220059/6.6/data/0-data/OceanSODA_ETHZ-v2025.OCADS.01-1982-2024.nc"
ds = xr.open_dataset(fsspec.open(url).open(), engine="h5netcdf")

In [2]:
# Reduce to 2 deg resolution
ds = ds.isel(lat=slice(None, None, 3), lon=slice(None, None, 3))

In [3]:
variables = list(ds.variables) 
keep = {'talk','dic','temperature','salinity'}
drop_list = [item for item in variables if item not in keep]
drop_list

['spco2',
 'sfco2',
 'ph_total',
 'ph_free',
 'hco3',
 'co3',
 'co2',
 'revelle_factor',
 'omega_ca',
 'omega_ar',
 'dic_uncert',
 'spco2_uncert',
 'ph_total_uncert',
 'omega_ca_uncert',
 'omega_ar_uncert',
 'talk_uncert',
 'sfco2_uncert',
 'lat',
 'lon',
 'time',
 'year']

In [4]:
ds = ds.drop_vars(drop_list)

In [5]:
ds = ds.load()

In [6]:
ds

<xarray.Dataset> Size: 59MB
Dimensions:      (time: 516, lat: 60, lon: 120)
Dimensions without coordinates: time, lat, lon
Data variables:
    talk         (time, lat, lon) float32 15MB nan nan ... 2.162e+03 2.161e+03
    dic          (time, lat, lon) float32 15MB nan nan nan nan ... nan nan nan
    temperature  (time, lat, lon) float32 15MB nan nan nan ... -1.671 -1.684
    salinity     (time, lat, lon) float32 15MB nan nan nan ... 29.45 29.46 29.5
Attributes:
    contact:      gregorl@ethz.ch
    author:       Luke Gregor
    institution:  ETH Zuerich
    version:      v2025.OCADS
    date:         2025-10-09
    description:  talk and pco2 (more accurately fco2) are estimated with two...
    changelog:    v2021d: Extended from 1982-2020; Now using: OISSTv2.1 for S...
    reference:    Gregor, L. and Gruber, N.: OceanSODA-ETHZ: A global gridded...
    source:       https://doi.org/10.25921/m5wx-ja34
    product:      OSETHZ-v2025.OCADS

In [7]:
ds.to_netcdf('./updated_files/coarsened_OceanSODA_dataset.nc')